# Tokenizer training on Colab (larger sample size)

The local machine only has ~7.6GB RAM, which forced SentencePiece training down to a
150,000-sentence sample (the unigram trainer's EM phase needs far more memory than
the raw sampled text — 5M sentences OOM-killed the process, 1M hit `std::bad_alloc`
under a 3GB cap). Colab's free tier gives ~12-13GB RAM — more room, but not unlimited:
a first attempt at 2M sentences OOM-crashed the Colab kernel too. The config cell below
now targets 1,000,000 sentences as the next attempt (untested at this RAM budget); drop
further to 500,000 if that still crashes.

**Steps:**
1. Runtime > Change runtime type > make sure you're on a CPU or standard runtime (no GPU/TPU needed).
2. Run all cells with `LANG = "hindi"`, download the 3 output files.
3. Change `LANG = "nepali"` at the top, Runtime > Restart session, run all cells again, download.
4. Bring the downloaded files back into the local repo (see final cell for exact destination paths).

In [ ]:
!pip install -q sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Set `LANG` to `"hindi"` or `"nepali"` and re-run the whole notebook once per language.

`DRIVE_DATA_ROOT` should point at the folder that contains `<LANG>/processed/train.txt`
and `<LANG>/processed/val.txt` on your Drive — adjust if your upload used a different
path than `projects/LMA-individual/data`.

In [ ]:
LANG = "hindi"  # or "nepali"

DRIVE_DATA_ROOT = "/content/drive/MyDrive/projects/LMA-individual/data"

VOCAB_SIZE = 32_000
# The unigram trainer's EM/seed-vocabulary phase (right after "Done! preprocessed N
# sentences...") is far more memory-hungry than the raw sampled text, and scales worse
# than linearly with sentence count:
#   - 1,000,000 sentences (292MB text) hit std::bad_alloc under a 3GB cap locally.
#   - 2,000,000 sentences (587MB text) OOM-crashed the Colab kernel outright at 12.7GB RAM.
# 1,000,000 has never actually been tried on Colab's larger RAM budget (only on the
# 3GB-capped local machine), so it's the next value to try — halving the failed 2M
# rather than guessing further. If this still crashes, drop to 500,000 next.
INPUT_SENTENCE_SIZE = 1_000_000

print(f"LANG={LANG}  VOCAB_SIZE={VOCAB_SIZE:,}  INPUT_SENTENCE_SIZE={INPUT_SENTENCE_SIZE:,}")

In [ ]:
import os

train_path = f"{DRIVE_DATA_ROOT}/{LANG}/processed/train.txt"
val_path = f"{DRIVE_DATA_ROOT}/{LANG}/processed/val.txt"

assert os.path.exists(train_path), f"train.txt not found at {train_path} — fix DRIVE_DATA_ROOT above"
assert os.path.exists(val_path), f"val.txt not found at {val_path} — fix DRIVE_DATA_ROOT above"

print("train.txt size (bytes):", os.path.getsize(train_path))
print("val.txt size (bytes):", os.path.getsize(val_path))

## Train

Same hyperparameters as the local `scripts/tokenizer/train_tokenizer.py` (unigram,
32k vocab, character_coverage=0.9995, shuffled sampling, standard special-token ids) —
only `input_sentence_size` differs. Writes to local Colab disk (not Drive) so results
are easy to grab with `files.download()` below.

In [ ]:
import sentencepiece as spm

model_prefix = f"/content/{LANG}_unigram_{VOCAB_SIZE // 1000}k"

spm.SentencePieceTrainer.train(
    input=train_path,
    model_prefix=model_prefix,
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=0.9995,
    input_sentence_size=INPUT_SENTENCE_SIZE,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    num_threads=os.cpu_count(),
)
print(f"wrote {model_prefix}.model / {model_prefix}.vocab")

## Evaluate on held-out val.txt

Same logic as `scripts/tokenizer/evaluate_tokenizer.py`: fertility, UNK rate, avg
chars/token, top tokens, and a few example tokenizations.

In [ ]:
import json
from collections import Counter

SAMPLE_LINES = 200_000

sp = spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")
unk_id = sp.unk_id()

lines = []
with open(val_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            lines.append(line)
        if len(lines) >= SAMPLE_LINES:
            break

total_words = total_tokens = total_chars = unk_count = 0
token_freq = Counter()

for line in lines:
    ids = sp.encode(line, out_type=int)
    total_words += len(line.split())
    total_tokens += len(ids)
    total_chars += len(line)
    unk_count += sum(1 for i in ids if i == unk_id)
    token_freq.update(ids)

fertility = total_tokens / total_words if total_words else 0.0
unk_rate = unk_count / total_tokens if total_tokens else 0.0
avg_chars_per_token = total_chars / total_tokens if total_tokens else 0.0

examples = []
for line in lines[:5]:
    pieces = sp.encode(line, out_type=str)
    examples.append({"text": line[:150], "num_tokens": len(pieces), "tokens": pieces[:40]})

stats = {
    "language": LANG,
    "model_file": os.path.basename(f"{model_prefix}.model"),
    "vocab_size": sp.vocab_size(),
    "eval_split": "val (held out, never used in training)",
    "eval_lines_sampled": len(lines),
    "eval_words": total_words,
    "eval_tokens": total_tokens,
    "fertility_tokens_per_word": round(fertility, 4),
    "unk_rate": round(unk_rate, 6),
    "avg_chars_per_token": round(avg_chars_per_token, 4),
    "top_20_most_frequent_tokens": [
        {"token": sp.id_to_piece(tid), "count": c}
        for tid, c in token_freq.most_common(20)
    ],
    "examples": examples,
}

stats_path = f"/content/{LANG}_tokenizer_stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(json.dumps(stats, indent=2, ensure_ascii=False))

## Download results

Downloads 3 files. Put them back into the local repo at:
- `<LANG>_unigram_32k.model` -> `<repo>/<LANG>/tokenizer/<LANG>_unigram_32k.model`
- `<LANG>_unigram_32k.vocab` -> `<repo>/<LANG>/tokenizer/<LANG>_unigram_32k.vocab`
- `<LANG>_tokenizer_stats.json` -> `<repo>/report/<LANG>_tokenizer_stats.json`

(overwriting the existing 150k-sample versions), then repeat the whole notebook with
`LANG = "nepali"`.

In [ ]:
from google.colab import files

files.download(f"{model_prefix}.model")
files.download(f"{model_prefix}.vocab")
files.download(stats_path)